# 02 - Point-in-time features + rules baseline

Manchester.Ai_IBM Z Datathon 2026 Project. Runs on IBM LinuxONE.

**The question we answer:** the moment a payment lands in an account, does it look like that account is being used as a money mule?

**The rule that keeps us honest:** every feature for a payment uses only transactions from *strictly before* that payment's timestamp. Nothing from the future leaks in, so the model only knows what a bank would know at that moment.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

SHARED = os.path.expanduser('~/shared')
t0 = time.time()
df = pd.read_pickle(f'{SHARED}/hi_small_clean.pkl')
print(df.shape, f'loaded in {time.time() - t0:.1f}s')

## 1. Scope the data
- **Drop self-transfers** (11.6% of rows, only 11 laundering): money moving within one account is not a mule inflow.
- **Drop 11 Sep onwards:** after 10 Sep normal activity stops and the few remaining rows are mostly laundering (a quirk of how the synthetic data was generated). Keeping them would make our results look far better than they really are.

In [ ]:
df = df[df['sender'] != df['receiver']]
df = df[df['timestamp'] < '2022-09-11']
df = df.sort_values('timestamp', kind='stable').reset_index(drop=True)
df['t'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds().astype('int64')

# One shared integer ID per account, used whether it is sending or receiving
acct = pd.factorize(pd.concat([df['sender'], df['receiver']], ignore_index=True))[0].astype('int64')
df['s_id'] = acct[:len(df)]
df['r_id'] = acct[len(df):]
print(df.shape, '| laundering:', int(df['is_laundering'].sum()), f"({df['is_laundering'].mean():.3%})")

## 2. The point-in-time engine
`window_stats` answers questions like *"how many payments did this account receive in the 24 hours before this payment?"* for all ~4.5M rows at once.

It sorts events by (account, time) and uses binary search, so each feature takes seconds rather than a slow per-account loop. Events at the same minute as the current payment are excluded, so there is no leakage.

In [ ]:
def window_stats(q_id, q_t, e_id, e_t, window=None, e_val=None):
    """For each query row: count (and optionally sum of e_val) of events for the same
    account with time in [q_t - window, q_t). window=None means all earlier history."""
    q_id = np.asarray(q_id, dtype='int64'); q_t = np.asarray(q_t, dtype='int64')
    e_id = np.asarray(e_id, dtype='int64'); e_t = np.asarray(e_t, dtype='int64')
    big = np.int64(max(q_t.max(), e_t.max()) + 1)
    ek = e_id * big + e_t
    order = np.argsort(ek, kind='stable')
    ek = ek[order]
    lo_t = np.zeros_like(q_t) if window is None else np.maximum(q_t - window, 0)
    r = np.searchsorted(ek, q_id * big + q_t, side='left')
    l = np.searchsorted(ek, q_id * big + lo_t, side='left')
    cnt = (r - l).astype('int32')
    if e_val is None:
        return cnt
    cs = np.concatenate([[0.0], np.cumsum(np.asarray(e_val, dtype='float64')[order])])
    return cnt, (cs[r] - cs[l]).astype('float32')

# Tiny sanity test on hand-made data: account 7 receives at t=0, 10, 10, 50
_c = window_stats(q_id=[7, 7, 7, 7], q_t=[0, 10, 10, 50], e_id=[7, 7, 7, 7], e_t=[0, 10, 10, 50], window=45)
assert list(_c) == [0, 1, 1, 2], _c   # same-minute events excluded; window [5, 50) at t=50 holds t=10, 10
print('window_stats sanity check passed')

## 3. Build features
Three groups, matching the pitch: **the payment**, **the receiving account** (the possible mule) and **the sender** (the possible recruiter).

In [ ]:
t0 = time.time()
H, D = 3600, 86400
s, r, t = df['s_id'].values, df['r_id'].values, df['t'].values
amt_paid = df['amount_paid'].values
amt_recv = df['amount_received'].values
f = pd.DataFrame(index=df.index)

# --- The payment itself ---
f['log_amount'] = np.log1p(amt_paid).astype('float32')
f['payment_format'] = df['payment_format'].cat.codes.astype('int8')
f['payment_currency'] = df['payment_currency'].cat.codes.astype('int8')
f['cross_currency'] = (df['payment_currency'].astype(str) != df['receiving_currency'].astype(str)).astype('int8')
f['cross_bank'] = (df['from_bank'] != df['to_bank']).astype('int8')
f['hour'] = df['timestamp'].dt.hour.astype('int8')

new_pair = (~df.duplicated(['s_id', 'r_id'])).values          # first time this sender has paid this receiver
f['new_pair'] = new_pair.astype('int8')
pair = pd.factorize(df['s_id'] * (acct.max() + 1) + df['r_id'])[0]
f['pair_prior_cnt'] = window_stats(pair, t, pair, t)

# --- The receiving account (possible mule) ---
f['r_in_cnt_all'], r_in_sum_all = window_stats(r, t, r, t, None, amt_recv)
f['r_in_cnt_24h'], r_in_sum_24h = window_stats(r, t, r, t, D, amt_recv)
f['r_in_cnt_1h'] = window_stats(r, t, r, t, H)
f['r_out_cnt_all'] = window_stats(r, t, s, t)                              # has it ever sent money?
f['r_out_cnt_24h'], r_out_sum_24h = window_stats(r, t, s, t, D, amt_paid)  # money leaving recently
f['r_distinct_senders'] = window_stats(r, t, r[new_pair], t[new_pair])
f['r_new_senders_24h'] = window_stats(r, t, r[new_pair], t[new_pair], D)  # fan-in
avg_in = np.where(f['r_in_cnt_all'] > 0, r_in_sum_all / np.maximum(f['r_in_cnt_all'], 1), np.nan)
f['amt_vs_r_avg'] = np.where(np.isnan(avg_in), -1, amt_recv / (avg_in + 1)).astype('float32')
f['r_passthrough_24h'] = (r_out_sum_24h / (r_in_sum_24h + 1)).astype('float32')

# --- The sender (possible recruiter / source) ---
f['s_out_cnt_all'] = window_stats(s, t, s, t)
f['s_out_cnt_24h'] = window_stats(s, t, s, t, D)
f['s_in_cnt_24h'] = window_stats(s, t, r, t, D)                            # money passing through the sender too
f['s_distinct_receivers'] = window_stats(s, t, s[new_pair], t[new_pair])
f['s_new_receivers_24h'] = window_stats(s, t, s[new_pair], t[new_pair], D) # fan-out

# --- Account age: time since first seen anywhere in the data ---
first_seen = pd.concat([pd.Series(t, index=s), pd.Series(t, index=r)]).groupby(level=0).min()
f['r_age_h'] = ((t - first_seen.reindex(r).values) / H).astype('float32')
f['s_age_h'] = ((t - first_seen.reindex(s).values) / H).astype('float32')

FEATURES = list(f.columns)
print(f'{len(FEATURES)} features for {len(f):,} payments in {time.time() - t0:.1f}s')
f.describe().T[['mean', 'min', 'max']]

## 4. Time-based split
Day 1 is warm-up only (every account looks brand new on day 1). We train on earlier days and test on later days, never a random split, so the test mimics the real future.

In [ ]:
day = df['timestamp'].dt.day
split = np.select([day == 1, day <= 6, day <= 8], ['warmup', 'train', 'val'], default='test')
data = f.copy()
data['split'] = split
data['y'] = df['is_laundering'].values.astype('int8')
for col in ['timestamp', 'sender', 'receiver', 'amount_paid']:
    data[col] = df[col].values
data['payment_format_name'] = df['payment_format'].astype(str).values

summary = data.groupby('split')['y'].agg(rows='size', laundering='sum', rate='mean')
summary.loc[['warmup', 'train', 'val', 'test']]

## 5. Rules baseline
What a simple hand-written bank rule would catch. The model must clearly beat this.

**Metric:** a bank can only review a small share of payments, so we flag the riskiest **1%** and ask: what share of all laundering payments are in that 1%? (recall at a 1% alert rate). We also report PR-AUC, which suits very rare positives.

In [ ]:
def evaluate(y, score, rate=0.01):
    y = np.asarray(y); score = np.asarray(score, dtype='float64')
    k = max(1, int(len(y) * rate))
    top = np.argsort(-score, kind='stable')[:k]
    return {
        'recall@1%': y[top].sum() / max(y.sum(), 1),
        'precision@1%': y[top].mean(),
        'pr_auc': average_precision_score(y, score),
        'base_rate': y.mean(),
    }

ach = int(np.where(df['payment_format'].cat.categories == 'ACH')[0][0]) if 'ACH' in df['payment_format'].cat.categories else -1

def rules_score(d):
    flags = (
        (d['payment_format'] == ach).astype(int)        # most laundering in this data moves by ACH
        + d['new_pair']                                  # first payment between these two accounts
        + (d['s_new_receivers_24h'] >= 3).astype(int)    # sender fanning out to new accounts
        + (d['r_new_senders_24h'] >= 2).astype(int)      # receiver collecting from new senders
        + (d['r_age_h'] < 24).astype(int)                # brand-new receiving account
    )
    return flags + 1e-3 * d['log_amount']                # tiny tie-breaker on amount

test = data[data['split'] == 'test']
baseline = evaluate(test['y'], rules_score(test))
pd.Series(baseline).round(4)

## 6. Save for the modelling notebook

In [ ]:
data.to_pickle(f'{SHARED}/features.pkl')
with open(f'{SHARED}/feature_list.json', 'w') as fh:
    json.dump({'features': FEATURES, 'baseline_test': {k: float(v) for k, v in baseline.items()}}, fh, indent=2)
print('Saved shared/features.pkl and shared/feature_list.json')